# 01. Data Assembly

Loads the Federal Reserve's 2026 severely adverse scenario and the historical
actuals for the same variables, joins them at the 2025 Q4 anchor point, and
restricts the projection window to the nine-quarter horizon.

Verifies that the resulting peak-to-trough declines in the house price index
and the commercial real estate price index match the Federal Reserve's stated
scenario severity.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

raw = Path("..")/"data"/"raw"/"fed_scenarios"
processed = Path("..")/"data"/"processed"

scenario_file = raw/"2026_Final_Supervisory_Severely_Adverse_Domestic.csv"
historic_file = raw/"2026_Final_Historic_Domestic.csv"

scenario = pd.read_csv(scenario_file)
historic = pd.read_csv(historic_file)

print(f"scenario: {scenario.shape[0]} rows, {scenario.shape[1]} columns")
print(f"historic: {historic.shape[0]} rows, {historic.shape[1]} columns")

scenario: 13 rows, 18 columns
historic: 200 rows, 18 columns


In [2]:
print(list(scenario.columns) == list(historic.columns))

True


In [3]:
scenario["Date"] = pd.PeriodIndex(scenario["Date"].str.replace(" ",""), freq="Q")
historic["Date"] = pd.PeriodIndex(historic["Date"].str.replace(" ",""), freq="Q")

print(historic["Date"].min(), "to", historic["Date"].max())
print(scenario["Date"].min(), "to", scenario["Date"].max())

1976Q1 to 2025Q4
2026Q1 to 2029Q1


In [4]:
combined = pd.concat([historic, scenario], ignore_index=True)
combined = combined.sort_values("Date").reset_index(drop=True)

print(combined.shape)
print(combined["Date"].min(), "to", combined["Date"].max())

(213, 18)
1976Q1 to 2029Q1


In [5]:
anchor = pd.Period("2025Q4", freq="Q")

index_vars = [
    "Dow Jones Total Stock Market Index (Level)",
    "House Price Index (Level)",
    "Commercial Real Estate Price Index (Level)",
]

verify_vars = [
    "House Price Index (Level)",
    "Commercial Real Estate Price Index (Level)",
]

def pct_col_name(level_col):
    """Map an index level column to its percent-change counterpart."""
    return level_col.replace(" (Level)", " (pct chg from 2025Q4)")

anchor_row = combined.loc[combined["Date"] == anchor, index_vars]
print(anchor_row)

     Dow Jones Total Stock Market Index (Level)  House Price Index (Level)  Commercial Real Estate Price Index (Level)
199                                     67501.5                      323.4                                       305.8


In [6]:
anchor_val = anchor_row.iloc[0]

for col in index_vars:
    pct_col = col.replace(" (Level)", " (pct chg from 2025Q4)")
    combined[pct_col] = (combined[col]/anchor_val[col] - 1) * 100

In [7]:
horizon_start = pd.Period("2026Q1", freq="Q")
horizon_end = pd.Period("2028Q1", freq="Q")

horizon = combined[
    (combined["Date"] >= horizon_start) & (combined["Date"] <= horizon_end)
].copy()

print(f"horizon: {horizon.shape[0]} quarters")

for col in verify_vars:
    pct_col = pct_col_name(col)
    trough = horizon[pct_col].min()
    trough_date = horizon.loc[horizon[pct_col].idxmin(), "Date"]
    print(f"{col}: {trough:.1f}% at {trough_date}")

horizon: 9 quarters
House Price Index (Level): -29.7% at 2027Q4
Commercial Real Estate Price Index (Level): -38.8% at 2027Q4


In [8]:
processed.mkdir(parents=True, exist_ok=True)

scenario_out = processed/"scenario_severely_adverse_2026.csv"
horizon.to_csv(scenario_out, index=False)

print(f"Written: {scenario_out}")
print(f"{horizon.shape[0]} rows, {horizon.shape[1]} columns")

Written: ..\data\processed\scenario_severely_adverse_2026.csv
9 rows, 21 columns


In [12]:
historic_out = processed/"macro_history.csv"
macro_history = combined[combined["Date"] <= anchor].copy()
macro_history.to_csv(historic_out, index=False)

print(f"Written: {historic_out}")
print(f"{macro_history.shape[0]} rows, {macro_history.shape[1]} columns")

Written: ..\data\processed\macro_history.csv
200 rows, 21 columns


## Summary

Scenario and historical macro variables joined at the 2025 Q4 anchor.

**Verification.** Peak-to-trough declines over the nine-quarter horizon,
measured from the 2025 Q4 anchor:

| Variable | Decline | Trough |
|---|---|---|
| House Price Index | -29.7% | 2027 Q4 |
| Commercial Real Estate Price Index | -38.8% | 2027 Q4 |

These match the Federal Reserve's stated scenario severity of approximately
30 percent and 39 percent respectively, confirming that the anchor point and
horizon filter are correctly applied.

**Outputs.**

| File | Contents | Rows |
|---|---|---|
| `data/processed/scenario_severely_adverse_2026.csv` | Projection window, 2026 Q1 to 2028 Q1 | 9 |
| `data/processed/macro_history.csv` | Historical actuals, 1976 Q1 to 2025 Q4 | 200 |